# 03 — Semi-Synthetic Benchmarks

**Goal:** Rigorously compare all seven panel estimators across four treatment patterns using *semi-synthetic* data derived from the [`basque`](https://en.wikipedia.org/wiki/Basque_Country_(autonomous_community)) dataset.

### What is a semi-synthetic experiment?

A fully synthetic experiment generates both the outcome panel *and* the treatment mask from scratch, so results depend on the DGP assumptions.  
A semi-synthetic experiment instead:

1. **Extracts a clean baseline M** from real data (never-treated rows or pre-treatment columns).
2. **Draws a synthetic treatment mask Z** from a chosen pattern.
3. **Injects a known effect $\tau^*$** — so we know the ground truth.
4. **Runs each estimator** and measures recovery: $\text{error} = |\tau^* - \hat{\tau}| / |\tau^*|$.

Because the baseline comes from real data, confounding structure is realistic; because $\tau^*$ is injected, evaluation is exact.

## Treatment patterns and estimator support

Not every estimator is valid under every pattern — the table below shows which combinations are benchmarked.

| Pattern | Description | Supported estimators |
|:---:|:---|:---|
| **IID** | Each (unit, period) cell independently treated with prob 0.2 | DC-PR, MC-NNM, CovPCA |
| **Block** | A random contiguous block of units × periods is treated | DC-PR, MC-NNM, CovPCA, DID, SDID, SC, RSC |
| **Staggered** | Units adopt treatment at different randomly drawn times | DC-PR, MC-NNM, CovPCA, DID, SDID |
| **Adaptive** | Treatment triggers when a unit's outcome falls below a lookback minimum | DC-PR, MC-NNM, CovPCA |

Method keys used internally by `run_experiment`:

| Key | Class |
|:---|:---|
| `DC_PR_auto_rank` | `DCPanelSolver` |
| `MC_NNM_CV` | `MCNNMPanelSolver` (with cross-validation) |
| `CovariancePCA` | `CovariancePCAPanelSolver` |
| `DID` | `DIDPanelSolver` |
| `SDID` | `SDIDPanelSolver` |
| `SC` | `OLSSCPanelSolver` |
| `RobustSyntheticControl` | `RSCPanelSolver` |

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from causaltensor.datasets import PanelDataset
from causaltensor.semi_synthetic import run_experiment, run_aa_test

# Canonical display order for methods and patterns
METHODS_ORDER  = ["DC_PR_auto_rank", "MC_NNM_CV", "CovariancePCA", "DID", "SDID", "SC", "RobustSyntheticControl"]
PATTERNS_ORDER = ["IID", "Block", "Staggered", "Adaptive"]

---
## Step 1 — Dataset overview

In [3]:
dataset_name = "basque"
panel = PanelDataset.from_builtin(dataset_name)

O = panel.O   # (N, T) float array
Z = panel.Z   # (N, T) binary treatment mask

print(f"Dataset : {dataset_name!r}")
print(f"Shape   : {O.shape[0]} units × {O.shape[1]} periods")
print(f"Years   : {int(panel.time_index[0])} – {int(panel.time_index[-1])}")
treated_mask = Z.any(axis=1)
print(f"Treated : {list(panel.unit_index[treated_mask])}")
print(f"Control : {(~treated_mask).sum()} never-treated units (baseline pool)")

Dataset : 'basque'
Shape   : 18 units × 43 periods
Years   : 1955 – 1997
Treated : ['Basque Country (Pais Vasco)']
Control : 17 never-treated units (baseline pool)


In [26]:
# Real treatment mask — the semi-synthetic experiment replaces this with synthetic Z
Z_hot = pd.DataFrame(Z * 1.0, index=panel.unit_index, columns=panel.time_index.astype(int))
fig = px.imshow(
    Z_hot,
    color_continuous_scale=["#e8e8e8", "#2980b9"],
    zmin=0, zmax=1,
    labels={"x": "Period", "y": "Unit", "color": "Treated"},
    title=f"Real treatment mask — {dataset_name!r}  (used to identify the control)",
    aspect="auto",
)
fig.update_coloraxes(showscale=False)
fig.update_traces(ygap=2)  # visible separator between rows
fig.update_layout(height=max(500, Z.shape[0] * 28), margin=dict(l=160, r=20, t=50, b=40))
fig.show()

---
## Step 2 — Semi-synthetic experiment

`run_experiment` builds the control-rows baseline `M`, then for each `(treatment_level, pattern, trial)` triple:
- draws a fresh synthetic `Z_syn`,
- injects `tau_star = treatment_level × mean(|M|)` into `M`,
- runs all valid estimators and records `tau_hat` and `error`.

Increase `N_TRIALS` (e.g. to 50) for tighter confidence bands.

In [14]:
N_TRIALS         = 10              # increase for publication-quality estimates
TREATMENT_LEVELS = [0.05, 0.1, 0.2] # fractions of mean(|M|) injected as tau_star

print("Running semi-synthetic experiment...")
df_exp = run_experiment(
    O, Z,
    treatment_levels=TREATMENT_LEVELS,
    n_trials=N_TRIALS,
    seed=42,
    verbose=False,
)
print(f"Done — {len(df_exp):,} rows ({df_exp['method'].nunique()} methods × "
      f"{df_exp['pattern'].nunique()} patterns × "
      f"{df_exp['treatment_level'].nunique()} levels × {N_TRIALS} trials)")
display(df_exp.head(10))

Running semi-synthetic experiment...


C:\Users\arush\MIT Dropbox\Arushi Jain\RA\Panel Inference\causaltensor\src\causaltensor\cauest\MCNNM.py:197: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum((valid_Ω)*((O-res.baseline)**2)) / np.sum(valid_Ω)


Done — 540 rows (7 methods × 4 patterns × 3 levels × 10 trials)


,method,pattern,treatment_level,trial,tau_star,tau_hat,error
0,DC_PR_auto_rank,IID,0.05,0,0.266048,0.154974,0.417498
1,MC_NNM_CV,IID,0.05,0,0.266048,0.234059,0.120239
2,CovariancePCA,IID,0.05,0,0.266048,0.048571,0.817435
3,DC_PR_auto_rank,IID,0.05,1,0.266048,0.294099,0.105434
4,MC_NNM_CV,IID,0.05,1,0.266048,0.272176,0.023032
5,CovariancePCA,IID,0.05,1,0.266048,0.345580,0.298939
6,DC_PR_auto_rank,IID,0.05,2,0.266048,0.244367,0.081493
7,MC_NNM_CV,IID,0.05,2,0.266048,0.299873,0.127139
8,CovariancePCA,IID,0.05,2,0.266048,0.269133,0.011596
9,DC_PR_auto_rank,IID,0.05,3,0.266048,0.279292,0.049779


### 2a — Mean error summary

$\text{error} = |\tau^* - \hat{\tau}| / |\tau^*|$.  Lower is better.  Grey cells = combination not benchmarked.

In [16]:
sub02 = df_exp[df_exp["treatment_level"] == 0.1]

pivot_err = (
    sub02
    .groupby(["method", "pattern"])["error"]
    .mean()
    .unstack("pattern")
    .reindex(index=METHODS_ORDER, columns=PATTERNS_ORDER)
)
print("Mean relative error  |tau* − tau_hat| / |tau*|   (treatment_level = 0.1)")
display(pivot_err.round(3))

Mean relative error  |τ★ − τ̂| / |τ★|   (treatment_level = 0.1)


pattern,IID,Block,Staggered,Adaptive
method,,,,
DC_PR_auto_rank,0.101,0.713,0.620,0.307
MC_NNM_CV,0.028,0.585,0.520,0.235
CovariancePCA,0.104,3.320,3.158,1.116
DID,NaN,0.880,0.811,NaN
SDID,NaN,0.434,0.280,NaN
SC,NaN,0.545,NaN,NaN
RobustSyntheticControl,NaN,0.607,NaN,NaN


In [ ]:
import plotly.graph_objects as go

z_vals = pivot_err.values

# Build annotation text (show value or blank for NaN)
text_vals = [
    [f"{v:.2f}" if not np.isnan(v) else "" for v in row]
    for row in z_vals
]

fig = go.Figure(go.Heatmap(
    z=z_vals,
    x=PATTERNS_ORDER,
    y=METHODS_ORDER,
    text=text_vals,
    texttemplate="%{text}",
    colorscale="RdYlGn_r",
    zmin=0, zmax=1,
    colorbar=dict(title="Mean error"),
))
fig.update_layout(
    title="Mean relative error by method × pattern  (treatment_level = 0.1)",
    xaxis_title="Pattern",
    yaxis_title="Method",
    height=420,
    margin=dict(l=180, r=60, t=60, b=60),
    yaxis=dict(autorange="reversed"),
)
fig.show()

### 2b — Error distribution by pattern (treatment level = 0.1)

Box plots show the spread of relative error across the 10 trials for each estimator.

In [17]:
fig = px.box(
    sub02.dropna(subset=["error"]),
    x="method",
    y="error",
    color="pattern",
    category_orders={"method": METHODS_ORDER, "pattern": PATTERNS_ORDER},
    labels={"error": "Relative error  |τ* − τ̂| / |τ*|", "method": "Estimator", "pattern": "Pattern"},
    title="Error distribution by estimator and pattern  (treatment_level = 0.1)",
)
fig.update_layout(
    height=480,
    xaxis_tickangle=-25,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(title="Pattern", orientation="h", y=-0.35, x=0.5, xanchor="center"),
)
fig.show()

### 2c — Effect size vs. accuracy

Smaller treatment levels (weak signals) make estimation harder.  
This chart shows mean error across all valid patterns as treatment level varies.

In [18]:
tl_df = (
    df_exp
    .groupby(["method", "treatment_level"])["error"]
    .mean()
    .reset_index()
)

fig = px.line(
    tl_df,
    x="treatment_level",
    y="error",
    color="method",
    markers=True,
    category_orders={"method": METHODS_ORDER},
    labels={
        "treatment_level": "Treatment level  (τ* / mean|M|)",
        "error": "Mean relative error  (pooled across patterns)",
        "method": "Estimator",
    },
    title="Estimation accuracy vs. effect size",
)
fig.update_layout(
    height=440,
    margin=dict(l=60, r=20, t=60, b=100),
    legend=dict(orientation="h", y=-0.35, x=0.5, xanchor="center", title="Estimator"),
)
fig.show()

---
## Step 3 — A/A test (false positive rate)

An **A/A test** checks whether an estimator falsely detects a treatment effect when there is none.  
We run estimators on the pure baseline `M` (no injection), and flag a *false positive* when
$|\hat{\tau}| / \mathrm{std}(M) > 0.05$.

A well-calibrated estimator should have a false positive rate close to zero.

In [19]:
print("Running A/A test...")
df_aa = run_aa_test(
    O, Z,
    n_trials=N_TRIALS,
    seed=42,
    verbose=False,
)
print(f"Done — {len(df_aa):,} rows")
display(df_aa.head(10))

Running A/A test...
Done — 180 rows


,method,pattern,baseline_type,trial,tau_hat,std_M,relative_tau,is_false_positive
0,DC_PR_auto_rank,IID,control,0,-0.110051,2.248228,0.048950,False
1,MC_NNM_CV,IID,control,0,-0.031989,2.248228,0.014229,False
2,CovariancePCA,IID,control,0,-0.217477,2.248228,0.096733,True
3,DC_PR_auto_rank,IID,control,1,0.009056,2.248228,0.004028,False
4,MC_NNM_CV,IID,control,1,0.015929,2.248228,0.007085,False
5,CovariancePCA,IID,control,1,0.055009,2.248228,0.024468,False
6,DC_PR_auto_rank,IID,control,2,-0.020940,2.248228,0.009314,False
7,MC_NNM_CV,IID,control,2,-0.004299,2.248228,0.001912,False
8,CovariancePCA,IID,control,2,0.007087,2.248228,0.003152,False
9,DC_PR_auto_rank,IID,control,3,-0.019394,2.248228,0.008626,False


In [20]:
# False positive rate (FPR) table
fpr_pivot = (
    df_aa
    .groupby(["method", "pattern"])["is_false_positive"]
    .mean()
    .unstack("pattern")
    .reindex(index=METHODS_ORDER, columns=PATTERNS_ORDER)
)
print("False positive rate  (|tau_hat| / std(M) > 0.05)")
display(fpr_pivot.round(3))

False positive rate  (|τ̂| / std(M) > 0.05)


pattern,IID,Block,Staggered,Adaptive
method,,,,
DC_PR_auto_rank,0.0,0.8,0.7,0.6
MC_NNM_CV,0.0,0.8,0.6,0.7
CovariancePCA,0.3,1.0,1.0,0.4
DID,NaN,0.8,0.8,NaN
SDID,NaN,0.6,0.3,NaN
SC,NaN,0.9,NaN,NaN
RobustSyntheticControl,NaN,0.6,NaN,NaN


In [22]:
fpr_vals = fpr_pivot.values

fpr_text = [
    [f"{v:.2f}" if not np.isnan(v) else "" for v in row]
    for row in fpr_vals
]

fig = go.Figure(go.Heatmap(
    z=fpr_vals,
    x=PATTERNS_ORDER,
    y=METHODS_ORDER,
    text=fpr_text,
    texttemplate="%{text}",
    colorscale="RdYlGn_r",
    zmin=0, zmax=1,
    colorbar=dict(title="FPR"),
))
fig.update_layout(
    title="A/A test: false positive rate by method × pattern",
    xaxis_title="Pattern",
    yaxis_title="Method",
    height=420,
    margin=dict(l=180, r=60, t=60, b=60),
    yaxis=dict(autorange="reversed"),
)
fig.show()

---
## Summary

| What we learned | |
|:---|:---|
| **Best all-around** | DC-PR and MC-NNM tend to be competitive across all four patterns. |
| **Pattern specialization** | SC and RSC are restricted to Block treatment; DID/SDID break under IID or Adaptive. |
| **Effect size matters** | All methods improve as the injected effect grows relative to outcome noise. |
| **A/A calibration** | Methods with low FPR are better calibrated for real experiments. |

### Next steps

- **`02_synthetic_dgp.ipynb`** — fully synthetic DGP experiments (vary N, T, rank, noise).
- **Inference** — add per-estimator standard errors (see the TODO in `01_real_observed_panels.ipynb`).
- **Custom methods** — pass your own `methods` dict to `run_experiment` to benchmark a new estimator.